### Imports
Standard data handling (`numpy`, `pandas`, `seaborn`), plus the sklearn/xgboost pieces used later: `train_test_split` for the train/validation split, `OrdinalEncoder` + `ColumnTransformer` for encoding categorical columns, `XGBRegressor` as the baseline model, and `r2_score`/`root_mean_squared_error` for evaluation (R2 is the competition's actual metric, RMSE is a more intuitive secondary check).

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, root_mean_squared_error

### Load raw data
Read the competition's train and test CSVs.

In [ ]:
df_train = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")

df_train

Preview the test set — note it has no `y` column, since predicting `y` is the task.

In [ ]:
df_test

### Split target from features
`ID` is just a row identifier with no predictive value, so it's dropped from the feature set `X`. `test_ID` is kept separately since it's needed to build the Kaggle submission file later.

In [ ]:
y = df_train['y']
X = df_train.drop(['ID', 'y'], axis = 1)

test_ID = df_test['ID']
X_test = df_test.drop(['ID'], axis = 1)

### Train/validation split
Holds out 20% of the training data as `X_val`/`y_val`, standing in for genuinely unseen data, so we can honestly measure generalization before ever touching the real `X_test`.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, random_state = 42)

### Exploring numeric columns
All 368 non-categorical columns turn out to be binary (0/1). This split is for exploration only — the actual `cat_cols`/`num_cols` used in the pipeline below are recomputed from `X_train` directly, not from this cell.

In [ ]:
num_features = X.select_dtypes(exclude = 'str')
num_features

### Exploring categorical columns
8 text columns (`X0`-`X8`), with cardinality ranging from 4 (`X4`) up to 47 (`X0`). `X0`, `X2`, and `X5` also contain category values in `test.csv` that never appear in `train.csv` — this is what drives the encoding choice below (ordinal encoding with an explicit fallback for unseen values, instead of one-hot encoding).

In [ ]:
cat_features = X.select_dtypes(include = 'str')
cat_features

### Build the preprocessing pipeline
`cat_cols`/`num_cols` are computed from `X_train` (not `X`), so nothing about `X_val`'s categories leaks into what the encoder learns.

Ordinal encoding (not one-hot) is used because: cardinality up to 47 would make one-hot very wide, and tree models don't assume any real order in the integer codes, so there's no downside. `handle_unknown='use_encoded_value', unknown_value=-1` safely handles categories in `X_val`/`X_test` that were never seen while fitting on `X_train`, without needing to peek at val/test to build the mapping.

In [ ]:
cat_cols = X_train.select_dtypes(include='str').columns.tolist()
num_cols = X_train.select_dtypes(exclude='str').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols),
    ]
)

### Fit and apply the preprocessing
`set_output(transform='pandas')` makes `.transform()` return a DataFrame with real column names instead of a bare numpy array. The preprocessor is fit on `X_train` only; `X_val` and `X_test` are only ever *transformed* with the mapping already learned from train, never used to fit it.

In [ ]:
preprocessor.set_output(transform='pandas')
preprocessor.fit(X_train)

X_train_encoded = preprocessor.transform(X_train)
X_val_encoded   = preprocessor.transform(X_val)
X_test_encoded  = preprocessor.transform(X_test)

### Baseline model
Default (untuned) `XGBRegressor` hyperparameters, so later feature engineering and tuning have an honest number to compare against. No feature scaling — unnecessary for tree models, which split one column at a time regardless of magnitude.

In [ ]:
baseline_model = XGBRegressor(random_state=42)
baseline_model.fit(X_train_encoded, y_train)

### Evaluate on validation set
Predicts on `X_val`, which was never used to fit the encoder or the model. R2 is the competition's actual scoring metric; RMSE is a more intuitive "average miss" in `y`'s own units, kept as a secondary sanity check.

In [ ]:
y_val_pred = baseline_model.predict(X_val_encoded)

r2 = r2_score(y_val, y_val_pred)
rmse = root_mean_squared_error(y_val, y_val_pred)

print(f"Validation R2:   {r2:.4f}")
print(f"Validation RMSE: {rmse:.4f}")

## Findings so far / what should inform next steps

**Baseline result**
- Default (untuned) `XGBRegressor` → Validation R2 = 0.4493, RMSE = 9.2583.
- Competition's actual metric is R2. Public leaderboard scores for this competition typically top out around 0.55-0.58, so 0.4493 is a believable, unremarkable starting point — room to improve via feature work and tuning, not a sign of a broken pipeline.

**Feature structure**
- 8 categorical (text) columns: `X0, X1, X2, X3, X4, X5, X6, X8`. Cardinality ranges from 4 (`X4`) to 47-49 (`X0`).
- 368 numeric columns, all binary (0/1) — no scaling needed for tree models regardless.
- `X0`, `X2`, `X5` have categories in `test.csv` that never appear in `train.csv` (6, 14, and 4 affected test rows respectively, out of 4209) — this is why ordinal encoding with `unknown_value=-1` was used instead of one-hot, and why the encoder is fit on `X_train` only (not `X`, not `X_test`) to avoid leaking validation/test category vocabulary into training.

**Known outlier**
- One training row (`ID=1770`) has `y=265.32`; every other row falls between 72 and 170. Because RMSE/R2 are both built from squared error, this single point can disproportionately influence the model's fit.
- Confirmed this row currently lands in `X_train` under `random_state=42`, not in `X_val` — so it isn't distorting the validation score directly, but may still be distorting what the model learns.
- **Next experiment:** drop this row from training only (leave val/test untouched) and re-fit the baseline to see whether R2 improves.

**Candidates for feature engineering / dimensionality reduction (not yet done)**
- Check the 368 binary columns for constant (zero-variance) or exact-duplicate columns — common in this specific dataset — and drop them; trees ignore uninformative columns anyway but removing them reduces noise/training time.
- Consider PCA or similar dimensionality reduction on the binary block *only if* redundancy/duplication turns out to be substantial, or if overfitting becomes visible (train score much higher than validation score) — not a given win for tree models, so should be tested against the baseline rather than applied by default.
- Any change (row removal, dropped columns, PCA, tuning) should be compared against the R2 = 0.4493 baseline above before being kept.
